# AB Image Pair Description Generator

Use GPT-5-mini to generate change descriptions and robot instructions for image pairs

## 1. Parameter Configuration

Configure all required parameters here

In [ ]:
# =====================================================
# Parameter section - modify parameters here
# =====================================================

# OpenAI API Key (required)
OPENAI_API_KEY = ""

# OpenAI model (gpt-4o, gpt-4o-mini, etc.)
MODEL = "gpt-5-mini"

# Input directory containing pair_* folders
INPUT_DIR = ""

# Maximum output tokens
MAX_OUTPUT_TOKENS = 8000

## 2. Import dependencies and set API key

In [ ]:
import os
import base64
import json
import re
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# Set API key
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

from openai import OpenAI

print("Dependencies imported successfully!")

## 3. Data structures and utility functions

In [ ]:
@dataclass
class ObjectSpec:
    """Object specification: includes ID, pixel coordinates, and world coordinates (world coordinates are only used for external distance calculations)"""
    object_id: str
    pixel_center_a: Tuple[int, int]
    pixel_center_b: Optional[Tuple[int, int]] = None
    world_coordinate_a: Optional[Tuple[float, float, float]] = None
    world_coordinate_b: Optional[Tuple[float, float, float]] = None
    move_distance_m: Optional[float] = None


def image_to_data_url(image_path: str) -> str:
    """
    Convert a local image to a data URL for the OpenAI API.
    Supported formats: PNG/JPEG/WEBP/GIF
    """
    p = Path(image_path)
    if not p.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    suffix = p.suffix.lower().lstrip(".")
    mime_map = {
        "png": "image/png",
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "webp": "image/webp",
        "gif": "image/gif",
    }
    if suffix not in mime_map:
        raise ValueError(f"Unsupported image type: .{suffix}")

    b64 = base64.b64encode(p.read_bytes()).decode("utf-8")
    return f"data:{mime_map[suffix]};base64,{b64}"


def save_json(path: str, obj: Dict[str, Any]) -> None:
    """Save JSON file"""
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def _euclidean_distance_m(
    a: Tuple[float, float, float],
    b: Tuple[float, float, float],
) -> float:
    """Compute 3D Euclidean distance (meters)"""
    dx = a[0] - b[0]
    dy = a[1] - b[1]
    dz = a[2] - b[2]
    return math.sqrt(dx * dx + dy * dy + dz * dz)


def _format_distance_m(d: float, decimals: int = 2) -> str:
    """
    Format the distance as natural English:
    "moved about <distance> m"

    Design notes:
    - The distance value is precisely computed externally; this function only handles language formatting
    - Keep fixed decimal places to avoid model-side rounding or rewriting
    - Returns a directly reusable string without additional model processing
    """
    if d < 0:
        d = 0.0
    return f"moved about {d:.{decimals}f} m"

## 4. Prompt Construction

In [ ]:
def build_prompt(objects: List[ObjectSpec]) -> str:

    # 1) ENEN Target objects：ENENEN id + pixel_center_A（ENENENEN）
    obj_entries: List[Dict[str, Any]] = []
    for o in objects:
        entry: Dict[str, Any] = {
            "id": o.object_id,
            "pixel_center_A": [int(o.pixel_center_a[0]), int(o.pixel_center_a[1])],
        }
        obj_entries.append(entry)

    # 2) Render the object list as JSON-like text (stable field order)
    obj_lines: List[str] = []
    for e in obj_entries:
        obj_lines.append(f'  {{"id":"{e["id"]}","pixel_center_A":{e["pixel_center_A"]}}}')
    obj_block = "[\n" + ",\n".join(obj_lines) + "\n]"

    # 3) Prompt：ENEN Image A；ENENENENobjectENEN“ENENEN/ENENENobject（ENENENEN）”
    return f"""
You are given ONE image of a scene: Image A (BEFORE the object(s) were removed).

The target objects listed below are EXACTLY the objects that are missing in the final scene (i.e., they were removed/moved away). 
Your job is to describe what is gone, and write a minimal robot instruction that would cause that disappearance.

Image resolution: 1024 × 768.
Pixel coordinate system (for reference only; INTERNAL USE ONLY):
- Origin (0, 0) is the top-left corner
- x increases to the right, y increases downward

You will receive a list of target objects with pixel centers in Image A.
Use the pixel centers ONLY to locate the correct objects. Do NOT mention pixel centers in your output.

CRITICAL CONSTRAINTS (MUST FOLLOW):
- Return ONLY valid JSON. No markdown. No extra text.
- The coordinate system and resolution are for internal localization ONLY. Do NOT mention them.
- Do NOT include any pixel coordinates or any other numbers in your output.
- Do NOT write any of: "(x=..., y=...)", "x=", "y=", "pixel", "coordinate", "resolution", "approx.", "approximately".
- Describe locations only using natural scene language (e.g., "left side of the counter", "near the stove", "next to the sink", "on the shelf").
- Do NOT mention any objects that are not in the target list.
- If multiple similar objects exist, disambiguate using visible attributes (e.g., color, material, shape), not numbers.

TASK (for EACH target object id):
1) Description:
   - Write ONE short sentence that conveys the object (as seen in Image A) is now gone from the scene.
   - You MAY add relative location details ONLY if it makes the description significantly clearer.

2) Robot instruction:
   - Write ONE concise, single-step imperative instruction that would remove/move away that object from the scene.
   - Keep it minimal (no extra steps).

OUTPUT FORMAT (STRICT JSON):
{{
  "description": [
    {{"id": "<object_id>", "text": ""}},
    ...
  ],
  "instructions": [
    {{"id": "<object_id>", "text": ""}},
    ...
  ]
}}

Target objects (pixel centers are for locating only):
{obj_block}
""".strip()

## 5. Response Parsing and Validation

In [ ]:
def parse_model_output_to_json(text: str) -> Dict[str, Any]:
    """
    Robust JSON parser:
    1) Try direct parsing
    2) Extract the {...} block and parse
    3) Fall back to legacy format parsing
    """
    text = text.strip()

    # 1) Direct parsing
    try:
        return json.loads(text)
    except Exception:
        pass

    # 2) Extract JSON object
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass

    # 3) Fall back to legacy format parsing
    desc_match = re.search(r"description\s*:\s*\[(.*?)\]", text, flags=re.DOTALL | re.IGNORECASE)
    inst_match = re.search(r"instructions\s*:\s*\[(.*?)\]", text, flags=re.DOTALL | re.IGNORECASE)
    if desc_match and inst_match:
        def _parse_list_body(body: str) -> List[str]:
            return re.findall(r"\"(.*?)\"", body, flags=re.DOTALL)

        desc_list = _parse_list_body(desc_match.group(1))
        inst_list = _parse_list_body(inst_match.group(1))
        return {
            "description": [{"id": str(i), "text": t} for i, t in enumerate(desc_list)],
            "instructions": [{"id": str(i), "text": t} for i, t in enumerate(inst_list)],
            "_warning": "Parsed legacy format; IDs are enumerated, not the original object IDs."
        }

    raise ValueError("Failed to parse model output as JSON. Raw text:\n" + text)


def validate_result(result: Dict[str, Any], object_ids: List[str]) -> None:
    """
    Validate that the result contains all objects.
    """
    desc = result.get("description", [])
    inst = result.get("instructions", [])
    if not isinstance(desc, list) or not isinstance(inst, list):
        raise ValueError("JSON schema error: 'description' and 'instructions' must be lists.")

    if len(desc) != len(inst):
        raise ValueError("JSON schema error: 'description' and 'instructions' length mismatch.")

    got_ids_desc = [x.get("id") for x in desc if isinstance(x, dict)]
    got_ids_inst = [x.get("id") for x in inst if isinstance(x, dict)]
    if got_ids_desc != got_ids_inst:
        raise ValueError("Order/ID mismatch between description and instructions.")

    missing = [oid for oid in object_ids if oid not in got_ids_desc]
    if missing:
        raise ValueError(f"Missing objects in output: {missing}")


print("Parsing and validation functions are ready!")

## 6. OpenAI API Call

In [ ]:
def run_remove(
    image_a_path: str,
    objects: List[ObjectSpec],
    model: str = "gpt-4o-mini",
    max_output_tokens: int = 800,
) -> Dict[str, Any]:
    client = OpenAI()

    prompt = build_prompt(objects)
    img_a = image_to_data_url(image_a_path)

    # buildENENENEN（ENENEN Image A）
    content = [
        {"type": "text", "text": prompt},
        {"type": "text", "text": "Image A:"},
        {"type": "image_url", "image_url": {"url": img_a}},
    ]

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": content,
            }
        ],
        max_completion_tokens=max_output_tokens,
    )

    # Extract response text
    text = resp.choices[0].message.content
    result = parse_model_output_to_json(text)

    object_ids = [o.object_id for o in objects]
    validate_result(result, object_ids)
    return result


print("API call function is ready!")

## 7. Metadata Loading and Batch Processing

In [ ]:
def load_objects_from_metadata(metadata_path: str) -> List[ObjectSpec]:
    data = json.loads(Path(metadata_path).read_text(encoding="utf-8"))
    coordinates = data.get("coordinates", {})

    out: List[ObjectSpec] = []
    for obj_id, coords in coordinates.items():
        pixel_a = coords.get("pixel_center_A")

        # Skip objects missing required coordinates
        if pixel_a is None:
            continue

        # Use the last segment of the object path as the short ID
        short_id = obj_id.split("/")[-1]
        out.append(
            ObjectSpec(
                object_id=short_id,
                pixel_center_a=tuple(pixel_a),
            )
        )

    return out


def process_pair_folder(
    pair_folder: str,
    model: str = "gpt-4o-mini",
    max_output_tokens: int = 800,
) -> Optional[Dict[str, Any]]:
    """
    processENEN pair ENENEN（remove）：
    - ENread A_rgb.png（ENENEN）
    - EN metadata.json read pixel_center_A
    - ENENENENoutput removed ENENENENEN result.json
    """
    pair_path = Path(pair_folder)

    # Skip if already generated
    output_file = pair_path / "result.json"
    if output_file.exists():
        print(f"  [SKIP] Existing result.json in {pair_folder}")
        return None

    # Check required files
    image_a = pair_path / "A_rgb.png"
    metadata_file = pair_path / "metadata.json"

    if not image_a.exists():
        print(f"  [SKIP] Missing A_rgb.png in {pair_folder}")
        return None
    if not metadata_file.exists():
        print(f"  [SKIP] Missing metadata.json in {pair_folder}")
        return None

    # Load objects from metadata.json
    objects = load_objects_from_metadata(str(metadata_file))

    if not objects:
        print(f"  [SKIP] No valid objects with pixel_center_A in {pair_folder}")
        return None

    print(f"  Found {len(objects)} objects with pixel_center_A: {[o.object_id for o in objects]}")

    # ENEN API ENEN remove ENEN
    result = run_remove(
        image_a_path=str(image_a),
        objects=objects,
        model=model,
        max_output_tokens=max_output_tokens,
    )

    # Save result
    output_file = pair_path / "result.json"
    save_json(str(output_file), result)
    print(f"  [OK] Saved: {output_file}")

    return result


def _find_pair_folders(input_path: Path) -> List[Path]:
    """Find all pair_* folders under input_path.

    Compatible with two directory layouts:
    - input_path/pair_0000/
    - input_path/<split>/pair_0000/（ENEN 0-19、20-39 EN）
    """

    # 1) First try direct subdirectories of the current directory (fastest)
    direct = sorted([d for d in input_path.iterdir() if d.is_dir() and d.name.startswith("pair_")])
    if direct:
        return direct

    # 2) Otherwise search recursively (supports one level higher directory)
    return sorted([p for p in input_path.rglob("pair_*") if p.is_dir()])


def process_all_pairs(
    input_dir: str,
    model: str = "gpt-4o-mini",
    max_output_tokens: int = 800,
) -> Dict[str, Any]:
    """Traverse and process all pair_* folders under input_dir (supports multi-level directories)."""
    input_path = Path(input_dir)

    if not input_path.is_dir():
        raise ValueError(f"Input directory does not exist: {input_dir}")

    pair_folders = _find_pair_folders(input_path)

    if not pair_folders:
        print(f"No pair_* folders found under: {input_dir}")
        return {"total": 0, "success": 0, "skipped": 0, "failed": 0}

    print(f"Found {len(pair_folders)} pair folders to process")
    print("-" * 50)

    stats = {"total": len(pair_folders), "success": 0, "skipped": 0, "failed": 0}

    for i, pair_folder in enumerate(pair_folders, 1):
        # Print relative paths to show which subdirectory each item belongs to
        try:
            rel = str(pair_folder.relative_to(input_path))
        except Exception:
            rel = pair_folder.name

        print(f"\n[{i}/{len(pair_folders)}] Processing: {rel}")
        try:
            result = process_pair_folder(
                str(pair_folder),
                model=model,
                max_output_tokens=max_output_tokens,
            )
            if result is not None:
                stats["success"] += 1
            else:
                stats["skipped"] += 1
        except Exception as e:
            print(f"  [ERROR] Failed: {e}")
            stats["failed"] += 1

    print("\n" + "=" * 50)
    print("Processing complete!")
    print(f"  Total: {stats['total']}")
    print(f"  Success: {stats['success']}")
    print(f"  Skipped: {stats['skipped']}")
    print(f"  Failed: {stats['failed']}")

    return stats


print("Batch processing functions are ready!")

## 8. Run Batch Processing

Run the cell below to process all image pairs

In [ ]:
# Start processing!
stats = process_all_pairs(
    input_dir=INPUT_DIR,
    model=MODEL,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

## 9. View Result Example (Optional)

In [ ]:
# View the first successfully processed result
from IPython.display import JSON, display

input_path = Path(INPUT_DIR)
pair_folders = sorted([d for d in input_path.iterdir() if d.is_dir() and d.name.startswith("pair_")])

for pair_folder in pair_folders:
    result_file = pair_folder / "result.json"
    if result_file.exists():
        print(f"Result from: {pair_folder.name}")
        result = json.loads(result_file.read_text(encoding="utf-8"))
        display(JSON(result))
        break
else:
    print("No result.json files found.")

In [ ]:
# Display Image A/B (optional, for visual comparison; does not affect the remove pipeline that infers from Image A only)
import matplotlib.pyplot as plt

if pair_folders:
    first_pair = pair_folders[0]

    img_a = plt.imread(first_pair / "A_rgb.png")

    # B may not exist (depends on your data layout); if missing, only show A
    b_path = first_pair / "B_rgb.png"
    img_b = plt.imread(b_path) if b_path.exists() else None

    if img_b is None:
        plt.figure(figsize=(8, 6))
        plt.imshow(img_a)
        plt.title("Image A")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        axes[0].imshow(img_a)
        axes[0].set_title("Image A")
        axes[0].axis("off")

        axes[1].imshow(img_b)
        axes[1].set_title("Image B")
        axes[1].axis("off")

        plt.tight_layout()
        plt.show()